## Load clean training data

In [ ]:
import pandas as pd

train_clean_dummies = pd.read_csv('train_test_clean/train_clean_dummies.csv')
train_clean_no_dummies = pd.read_csv('train_test_clean/train_clean_no_dummies.csv')

## Get time series split for training data (dummies / no dummies)

In [ ]:
from scripts.time_series_split import get_moving_windows_split_train, print_folds_shape

target_years = [2015, 2016, 2017, 2018]

train_dummies = get_moving_windows_split_train(train_clean_dummies, target_years, dummies='Y')
train = get_moving_windows_split_train(train_clean_no_dummies, target_years, dummies='N')

print_folds_shape(train_dummies)
print_folds_shape(train)

## Preprocess training data for models

In [ ]:
from scripts.preprocess_for_models import transform_datatype_for_ebm, sanitize_column_names_for_xgb, apply_preprocessing_to_folds

train_ebm = apply_preprocessing_to_folds(train, transform_datatype_for_ebm)
train_xgb = apply_preprocessing_to_folds(train_dummies, sanitize_column_names_for_xgb)

## Fit and evaluate logistic regression on training data

In [ ]:
from scripts.fit_evaluate_models import dataset_info, fit_lr, get_results_lr

folds_models_lr = fit_lr(train_dummies, dataset_info, parameter=0.1)
weights_lr, scores_lr = get_results_lr(folds_models_lr, dataset_info)

print(scores_lr)

## Fit and evaluate random forest on training data

In [ ]:
from scripts.fit_evaluate_models import fit_model, get_results
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    'n_estimators': 500,
    'max_depth': 10,
    'max_features': 50
}

folds_rf = fit_model(train_dummies, RandomForestClassifier, rf_params)
scores_rf = get_results(folds_rf)

print(scores_rf)

## Fit and evaluate gradient boosting on training data

In [ ]:
from scripts.fit_evaluate_models import fit_model, get_results
from sklearn.ensemble import GradientBoostingClassifier

gb_params = {
    'n_estimators': 500, 
    'learning_rate': 0.1, 
    'max_depth': 6
    }

# note: for n_estimators = 500, fitting takes approx. 30 minutes, or even more; for 100, just a couple of minutes.

folds_gb = fit_model(train_dummies, GradientBoostingClassifier, gb_params)
scores_gb = get_results(folds_gb)

print(scores_gb)

## Fit and evaluate XGBoost on training data

In [ ]:
from scripts.fit_evaluate_models import fit_model, get_results
import xgboost as xgb

xgb_params = {
    'max_depth': 6, 
    'learning_rate': 0.1 , 
    'reg_lambda': 5, 
    'objective': 'binary:logistic', 
    'n_estimators': 500    
    }

folds_xgb = fit_model(train_xgb, xgb.XGBClassifier, xgb_params)
scores_xgb = get_results(folds_xgb)

print(scores_xgb)

## Fit and evaluate EBM on training data

In [ ]:
from scripts.fit_evaluate_models import fit_model, get_results
from interpret.glassbox import ExplainableBoostingClassifier
from scripts.fit_evaluate_models import save_folds_with_models

EBM_params = {'interactions': 30, 'outer_bags': 9, 'learning_rate': 0.0014, 'min_samples_leaf': 2, 'max_leaves': 3}

folds_ebm = fit_model(train_ebm, ExplainableBoostingClassifier, EBM_params)
scores_ebm = get_results(folds_ebm)

print(scores_ebm)

save_folds_with_models(folds_ebm, 'models/folds_ebm.joblib')

## Save and plot AUC for all models over training range

In [ ]:
from scripts.fit_evaluate_models import combine_model_scores, plot_model_scores

combined_scores_df = combine_model_scores(scores_lr, scores_rf, scores_xgb, scores_gb, scores_ebm)

combined_scores_df.to_csv("results/model_scores.csv", index=False)

years = [2015, 2016, 2017, 2018]

plot_model_scores(combined_scores_df, years=years)


## Load test data (2018-train, 2019-test)

In [ ]:
import pandas as pd

test_clean_dummies = pd.read_csv('train_test_clean/test_clean_dummies.csv')
test_clean_no_dummies = pd.read_csv('train_test_clean/test_clean_no_dummies.csv')


## Get split for test data

In [ ]:
from scripts.time_series_split import get_moving_windows_split_train, print_folds_shape

target_years_test = [2019]

test_dummies = get_moving_windows_split_train(test_clean_dummies, target_years_test, dummies='Y')
test = get_moving_windows_split_train(test_clean_no_dummies, target_years_test, dummies='N')

print_folds_shape(test_dummies)
print_folds_shape(test)

## Preprocess test data for models

In [ ]:
from scripts.preprocess_for_models import transform_datatype_for_ebm, sanitize_column_names_for_xgb, apply_preprocessing_to_folds

test_ebm = apply_preprocessing_to_folds(test, transform_datatype_for_ebm)
test_xgb = apply_preprocessing_to_folds(test_dummies, sanitize_column_names_for_xgb)

## Fit and evaluate models on test data

In [ ]:
# LOGISTIC REGRESSION

from scripts.fit_evaluate_models import dataset_info, fit_lr, get_results_lr

test_fold_lr = fit_lr(test_dummies, dataset_info, parameter=0.1)
test_weights_lr, test_score_lr = get_results_lr(test_fold_lr, dataset_info)

# RANDOM FOREST

from scripts.fit_evaluate_models import fit_model, get_results
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    'n_estimators': 500,
    'max_depth': 10,
    'max_features': 50,
    'random_state': 42 
}

test_fold_rf = fit_model(test_dummies, RandomForestClassifier, rf_params)
test_score_rf = get_results(test_fold_rf)

## GRADIENT BOOSTING

from sklearn.ensemble import GradientBoostingClassifier

gb_params = {
    'n_estimators': 500, 
    'learning_rate': 0.1, 
    'max_depth': 6,
    'random_state': 42
    }

test_fold_gb = fit_model(test_dummies, GradientBoostingClassifier, gb_params)
test_score_gb = get_results(test_fold_gb)

# EXTREME GRADIENT BOOSTING

import xgboost as xgb

xgb_params = {
    'max_depth': 6, 
    'learning_rate': 0.1 , 
    'reg_lambda': 5, 
    'objective': 'binary:logistic', 
    'n_estimators': 500,
    'random_state': 42    
    }

test_fold_xgb = fit_model(test_xgb, xgb.XGBClassifier, xgb_params)
test_score_xgb = get_results(test_fold_xgb)

# EXPLAINABLE BOOSTING MACHINE

from interpret.glassbox import ExplainableBoostingClassifier

EBM_params = {'interactions': 30, 'outer_bags': 9, 'learning_rate': 0.0014, 'min_samples_leaf': 2, 'max_leaves': 3, 'random_state': 42}

test_fold_ebm = fit_model(test_ebm, ExplainableBoostingClassifier, EBM_params)
test_score_ebm = get_results(test_fold_ebm)


## Save and plot AUC for all models on test set

In [ ]:
from scripts.fit_evaluate_models import combine_model_scores, plot_model_scores

combined_test_scores_df = combine_model_scores(test_score_lr, test_score_rf, test_score_xgb, test_score_gb, test_score_ebm)

combined_test_scores_df.to_csv("results/model_scores_test.csv", index=False)

plot_model_scores(combined_test_scores_df, [2019])

## Combined Plot Train Test

In [ ]:
from scripts.fit_evaluate_models import plot_model_scores_total
import pandas as pd

model_train_scores = pd.read_csv('results/model_scores.csv')

model_test_scores = pd.read_csv('results/model_scores_test.csv')

model_train_test_scores = pd.concat([model_train_scores, model_test_scores], ignore_index=True)

test_years = [2015, 2016, 2017, 2018, 2019]

plot_model_scores_total(model_train_test_scores, test_years)